# RECIP-E — Interpretador (front-end) em Scheme

Projeto da disciplina **MC346 — Paradigmas de Programação** (UNICAMP).

Este é o **notebook único** do grupo: todas as etapas do processamento da DSL
**RECIP-E** vivem aqui dentro, em ordem de pipeline — léxico → sintático →
impressão/validação. Kernel: **Calysto Scheme**.

> **Estado: Dia 2 (Dom 28/06)** — a seção **1. Lexer** (Pessoa A) está completa,
> incluindo agora a **indentação** (`NEWLINE` / `INDENT` / `DEDENT`). As seções
> **2. Parser** (Pessoa B) e **3. Pretty-print + Validação** (Pessoa C) estão
> reservadas.
>
> Para conferir tudo: **Kernel → Restart & Run All**.

## 0. Setup / Contrato do token

Tudo no projeto conversa por meio de **tokens**. Combinamos um formato simples:

```
token = (tipo lexema linha)
```

- **tipo** — um símbolo: `NUMBER STRING IDENT KEYWORD VERB UNIT PREP PONT` e os
  tokens de estrutura `NEWLINE INDENT DEDENT` (gerados pela indentação).
- **lexema** — o texto original (string); vazio `""` para `NEWLINE/INDENT/DEDENT`.
- **linha** — número da linha no código-fonte (útil para mensagens de erro).

As funções `faz-token`, `tipo-tok`, `lexema-tok`, `linha-tok` são a **interface**
que a Pessoa B (parser) vai usar para ler os tokens.

In [ ]:
;; ---- Contrato do token: (tipo lexema linha) ----
;; tipos possiveis:
;;   NUMBER STRING IDENT KEYWORD VERB UNIT PREP PONT   (conteudo)
;;   NEWLINE INDENT DEDENT                             (estrutura/indentacao)
(define (faz-token tipo lexema linha) (list tipo lexema linha))
(define (tipo-tok   t) (car   t))
(define (lexema-tok t) (cadr  t))
(define (linha-tok  t) (caddr t))

;; ---- Tabelas de palavras reservadas da linguagem ----
(define palavras-bloco
  (list "RECEITA" "INGREDIENTES" "UTENSILIOS" "METODOS"))

(define palavras-controle
  (list "ESTE" "é" "AGUARDAR" "VERIFICAR" "SE" "ESTA"
        "CONTINUAR" "SENAO" "AO_MESMO_TEMPO"
        "GRAUS" "MINUTOS" "HORAS" "SEGUNDOS"))

(define preposicoes
  (list "EM" "COM" "USANDO" "A" "POR" "ATE" "NO_ESTILO"))

(define verbos
  (list "ADICIONAR" "MISTURAR" "BATER" "TEMPERAR" "ASSAR" "FRITAR"
        "COZINHAR" "REFOGAR" "GRELHAR" "CORTAR" "RALAR" "ESCORRER"
        "COLOCAR" "DECORAR"))

(define unidades
  (list "g" "kg" "ml" "l" "unidade" "colher" "colher_cha"
        "xcara" "pitada" "a_gosto" "faca"))

;; verifica se uma palavra esta numa lista (devolve #t ou #f)
(define (esta-em? palavra lista)
  (if (member palavra lista) #t #f))

;; descobre o tipo de uma palavra ja lida pelo lexer
(define (classifica palavra)
  (cond
    ((esta-em? palavra palavras-bloco)    'KEYWORD)
    ((esta-em? palavra palavras-controle) 'KEYWORD)
    ((esta-em? palavra preposicoes)       'PREP)
    ((esta-em? palavra verbos)            'VERB)
    ((esta-em? palavra unidades)          'UNIT)
    (else                                 'IDENT)))

## 1. Lexer (Pessoa A)

O lexer transforma o texto da receita numa **lista de tokens**. Ele reconhece
números, strings, identificadores e palavras reservadas, ignora comentários
(`//` e `/* */`) e — desde o Dia 2 — entende a **indentação**, emitindo
`NEWLINE` no fim de cada linha e `INDENT`/`DEDENT` quando o recuo aumenta ou
diminui.

### 1.1 Predicados auxiliares

Funções pequenas para classificar caracteres. `TAB` e `RET` são definidos via
`integer->char` para não depender de literais de caractere especiais.

In [ ]:
;; alguns caracteres "invisiveis"
(define TAB (integer->char 9))    ; tabulacao
(define RET (integer->char 13))   ; carriage return

;; e um digito de 0 a 9?  (Calysto Scheme nao tem char>=? / char<=?)
(define (eh-digito? c)
  (let ((code (char->integer c)))
    (and (>= code 48) (<= code 57))))

;; e uma letra (A-Z, a-z), underline ou letra acentuada (é, ç, ...)?
(define (eh-letra? c)
  (let ((code (char->integer c)))
    (or (and (>= code 65) (<= code 90))     ; A-Z
        (and (>= code 97) (<= code 122))    ; a-z
        (= code 95)                         ; _
        (>= code 128))))                    ; acentuadas

;; e um espaco em branco (sem contar a quebra de linha)?
(define (eh-espaco? c)
  (or (char=? c #\space) (char=? c TAB) (char=? c RET)))

### 1.2 Remoção de comentários

RECIP-E aceita comentário de linha `// ...` e de bloco `/* ... */`. Antes de
tokenizar, trocamos os comentários por nada — **mas preservamos as quebras de
linha**, para o número de linha continuar certo — e tomamos cuidado para não
apagar um `//` que esteja **dentro de uma string**.

In [ ]:
;; remove comentarios // e /* */, preservando quebras de linha e strings
(define (remove-comentarios s)
  (let ((n (string-length s)))
    (define (loop i estado acc)
      (if (>= i n)
          (list->string (reverse acc))
          (let ((c (string-ref s i)))
            (cond
              ;; ----- texto normal -----
              ((eq? estado 'normal)
               (cond
                 ((char=? c #\")
                  (loop (+ i 1) 'string (cons c acc)))
                 ((and (char=? c #\/) (< (+ i 1) n)
                       (char=? (string-ref s (+ i 1)) #\/))
                  (loop (+ i 2) 'linha acc))
                 ((and (char=? c #\/) (< (+ i 1) n)
                       (char=? (string-ref s (+ i 1)) #\*))
                  (loop (+ i 2) 'bloco acc))
                 (else
                  (loop (+ i 1) 'normal (cons c acc)))))
              ;; ----- dentro de uma string -----
              ((eq? estado 'string)
               (if (char=? c #\")
                   (loop (+ i 1) 'normal (cons c acc))
                   (loop (+ i 1) 'string (cons c acc))))
              ;; ----- comentario de linha: pula ate a quebra -----
              ((eq? estado 'linha)
               (if (char=? c #\newline)
                   (loop (+ i 1) 'normal (cons c acc))
                   (loop (+ i 1) 'linha acc)))
              ;; ----- comentario de bloco: pula ate */ -----
              ((eq? estado 'bloco)
               (cond
                 ((and (char=? c #\*) (< (+ i 1) n)
                       (char=? (string-ref s (+ i 1)) #\/))
                  (loop (+ i 2) 'normal acc))
                 ((char=? c #\newline)
                  (loop (+ i 1) 'bloco (cons c acc)))   ; mantem a linha
                 (else
                  (loop (+ i 1) 'bloco acc))))
              (else
               (loop (+ i 1) 'normal acc))))))
    (loop 0 'normal '())))

### 1.3 Leitura de números, identificadores e strings

Cada leitor recebe a string `s`, a posição inicial `i` e um limite `fim`, e
devolve a **posição final** do token (onde ele termina).

In [ ]:
;; le um numero (inteiro ou decimal); devolve a posicao final
(define (le-numero s i fim)
  (cond
    ((>= i fim) i)
    ((eh-digito? (string-ref s i)) (le-numero s (+ i 1) fim))
    ((and (char=? (string-ref s i) #\.) (< (+ i 1) fim)
          (eh-digito? (string-ref s (+ i 1))))
     (le-numero s (+ i 2) fim))
    (else i)))

;; le um identificador (letras, digitos, underline); devolve a posicao final
(define (le-ident s i fim)
  (if (and (< i fim)
           (or (eh-letra? (string-ref s i))
               (eh-digito? (string-ref s i))))
      (le-ident s (+ i 1) fim)
      i))

;; le uma string entre aspas; i aponta DEPOIS da aspa de abertura.
;; devolve a posicao logo apos a aspa de fechamento
(define (le-string s i fim)
  (cond
    ((>= i fim) fim)
    ((char=? (string-ref s i) #\") (+ i 1))
    (else (le-string s (+ i 1) fim))))

### 1.4 Indentação: `NEWLINE`, `INDENT`, `DEDENT`

Em RECIP-E o **recuo define o escopo** (não há chaves). O lexer processa o texto
**linha a linha** mantendo uma *pilha de níveis de indentação* (em número de
espaços; `tab` vale 4):

- recuo **maior** que o topo da pilha → empilha e emite **`INDENT`**;
- recuo **menor** → desempilha emitindo um **`DEDENT`** por nível fechado;
- recuo **igual** → nada;
- fim de cada linha com conteúdo → **`NEWLINE`**;
- linhas em branco (ou que só tinham comentário) são **ignoradas**;
- no fim do arquivo, fecha os níveis abertos com `DEDENT`.

> **Observação:** o arquivo deve usar recuo **consistente** (múltiplos de 4
> espaços). O alinhamento estético com espaços variáveis dos exemplos do PDF
> (ex.: números à direita) não é suportado nesta versão.

In [ ]:
;; indice da proxima quebra de linha a partir de i (ou n se nao houver)
(define (fim-da-linha s i n)
  (cond ((>= i n) n)
        ((char=? (string-ref s i) #\newline) i)
        (else (fim-da-linha s (+ i 1) n))))

;; primeiro indice >= i que NAO e espaco/tab (sem passar de fim)
(define (pula-brancos s i fim)
  (if (and (< i fim) (eh-espaco? (string-ref s i)))
      (pula-brancos s (+ i 1) fim)
      i))

;; "peso" da indentacao a partir de i: espaco = 1, tab = 4; para no 1o nao-branco
(define (peso-indentacao s i fim)
  (if (>= i fim)
      0
      (let ((c (string-ref s i)))
        (cond ((char=? c #\space) (+ 1 (peso-indentacao s (+ i 1) fim)))
              ((char=? c TAB)     (+ 4 (peso-indentacao s (+ i 1) fim)))
              (else 0)))))

;; ajusta a pilha para o novo nivel, gerando INDENT/DEDENT.
;; devolve um par (nova-pilha . novo-acc)
(define (ajusta-indent nivel pilha linha acc)
  (let ((topo (car pilha)))
    (cond
      ((> nivel topo)
       (cons (cons nivel pilha)
             (cons (faz-token 'INDENT "" linha) acc)))
      ((< nivel topo)
       (ajusta-indent nivel (cdr pilha) linha
                      (cons (faz-token 'DEDENT "" linha) acc)))
      (else
       (cons pilha acc)))))

;; no fim do arquivo, emite um DEDENT para cada nivel acima do nivel base (0)
(define (fecha-todos pilha linha acc)
  (if (or (null? pilha) (= (car pilha) 0))
      acc
      (fecha-todos (cdr pilha) linha
                   (cons (faz-token 'DEDENT "" linha) acc))))

### 1.5 A função `tokeniza`

`tokens-da-linha` quebra o conteúdo de **uma** linha em tokens; `tokeniza` junta
tudo: remove comentários, percorre linha a linha cuidando da indentação e
devolve a lista final de tokens. `mostra-tokens` só ajuda a visualizar.

In [ ]:
;; tokeniza o conteudo de UMA linha, do indice a ate b (exclusivo)
(define (tokens-da-linha s a b linha acc)
  (if (>= a b)
      acc
      (let ((c (string-ref s a)))
        (cond
          ((eh-espaco? c)
           (tokens-da-linha s (+ a 1) b linha acc))
          ((eh-digito? c)
           (let ((j (le-numero s a b)))
             (tokens-da-linha s j b linha
               (cons (faz-token 'NUMBER (substring s a j) linha) acc))))
          ((char=? c #\")
           (let ((j (le-string s (+ a 1) b)))
             (tokens-da-linha s j b linha
               (cons (faz-token 'STRING (substring s (+ a 1) (- j 1)) linha) acc))))
          ((eh-letra? c)
           (let* ((j (le-ident s a b))
                  (palavra (substring s a j)))
             (tokens-da-linha s j b linha
               (cons (faz-token (classifica palavra) palavra linha) acc))))
          ((char=? c #\,)
           (tokens-da-linha s (+ a 1) b linha (cons (faz-token 'PONT "," linha) acc)))
          ((char=? c #\:)
           (tokens-da-linha s (+ a 1) b linha (cons (faz-token 'PONT ":" linha) acc)))
          (else
           (tokens-da-linha s (+ a 1) b linha acc))))))

;; transforma o texto-fonte na lista de tokens (com NEWLINE/INDENT/DEDENT)
(define (tokeniza fonte)
  (let* ((s (remove-comentarios fonte))
         (n (string-length s)))
    (define (loop i linha pilha acc)
      (if (>= i n)
          (reverse (fecha-todos pilha linha acc))
          (let* ((fl  (fim-da-linha s i n))      ; fim da linha (quebra ou n)
                 (ini (pula-brancos s i fl)))     ; inicio do conteudo
            (if (>= ini fl)
                ;; linha em branco (ou so comentario): ignora
                (loop (+ fl 1) (+ linha 1) pilha acc)
                ;; linha com conteudo
                (let* ((nivel (peso-indentacao s i fl))
                       (par   (ajusta-indent nivel pilha linha acc))
                       (pilha2 (car par))
                       (acc2   (cdr par))
                       (acc3   (tokens-da-linha s ini fl linha acc2))
                       (acc4   (cons (faz-token 'NEWLINE "" linha) acc3)))
                  (loop (+ fl 1) (+ linha 1) pilha2 acc4))))))
    (loop 0 1 (list 0) '())))

;; mostra os tokens, um por linha (so para visualizar)
(define (mostra-tokens tokens)
  (for-each
    (lambda (t)
      (display (tipo-tok t))   (display "  ")
      (display (lexema-tok t)) (display "   (linha ")
      (display (linha-tok t))  (display ")")
      (newline))
    tokens))

### 1.6 Testes manuais

In [ ]:
;; uma linha simples: o comentario some e sobra um NEWLINE no fim
(mostra-tokens (tokeniza "10 g sal // isto e um comentario"))

In [ ]:
;; metadados com string:  NOME -> IDENT, : -> PONT, "..." -> STRING
(mostra-tokens (tokeniza "NOME: \"OmeleteComQueijo\""))

In [ ]:
;; comentario de bloco ocupando linhas inteiras: elas somem,
;; e a numeracao de linha do que sobra continua correta (linha 3)
(mostra-tokens
  (tokeniza "/* receita
   de teste */
ADICIONAR ovo EM tigela"))

In [ ]:
;; indentacao aninhada: repare nos INDENT/DEDENT e NEWLINE
(mostra-tokens
  (tokeniza "METODOS
    ADICIONAR ovo EM tigela
    AO_MESMO_TEMPO
        ADICIONAR oleo EM frigideira
        AGUARDAR 1 MINUTOS
    COLOCAR ovo EM prato"))

## 2. Parser

Transforma a lista de tokens em uma **AST** (árvore sintática abstrata)
representada como **listas com tag** — `(tipo ...)`.

### 2.1 Schema da AST

```
(programa <receita-ou-#f> <ingredientes> <utensilios> <metodos>)
(receita        ((campo NOME "...") ...))
(ingredientes   (<ingrediente> ...))
(utensilios     (<utensilio> ...))
(metodos        (<instrucao> ...))

(ingrediente    <qty> <unidade> <nome> <sub-receita?>)
(utensilio      <qty> <nome>)
(verb-cmd       <verbo> <args> <em> <com> <a> <por> <ate> <estilo>)
(arg            <nome> <usando-qty> <usando-unidade>)
(este-e         <nome>)
(aguardar       <qty> <unidade-tempo>)
(verificar      <alvo> <estado> <then-instrs> <senao-instrs-ou-#f>)
(ao-mesmo-tempo <ramo> ...)   ; ramo = lista de instruções
(continuar)
```

Campos opcionais ausentes são representados por `#f`.

### 2.2 Cursor de tokens

Encapsula a lista de tokens com `peek` (espia o próximo) e `advance`
(consome e devolve). Usa um *closure* com `set!` para evitar dependência
de `set-car!` (irregular em Calysto Scheme).

In [ ]:
;; cursor: encapsula a lista de tokens
(define (make-cursor tokens)
  (let ((toks tokens))
    (lambda (op)
      (cond
        ((eq? op 'peek)
         (if (null? toks) (faz-token 'EOF "" 0) (car toks)))
        ((eq? op 'advance)
         (if (null? toks)
             (faz-token 'EOF "" 0)
             (let ((t (car toks))) (set! toks (cdr toks)) t)))
        (else (error "cursor: op desconhecida"))))))

(define (peek c)    (c 'peek))
(define (advance c) (c 'advance))
(define (at? c tipo) (eq? (tipo-tok (peek c)) tipo))
(define (at-kw? c lex)
  (and (at? c 'KEYWORD) (string=? (lexema-tok (peek c)) lex)))
(define (at-prep? c lex)
  (and (at? c 'PREP)    (string=? (lexema-tok (peek c)) lex)))
(define (at-pont? c lex)
  (and (at? c 'PONT)    (string=? (lexema-tok (peek c)) lex)))

;; ---- erros ----
(define (erro-parse c msg)
  (let ((t (peek c)))
    (error (string-append "[parse] " msg
             " -- achei " (symbol->string (tipo-tok t))
             " '" (lexema-tok t) "'"
             " na linha " (number->string (linha-tok t))))))

(define (expect c tipo)
  (if (at? c tipo)
      (let ((t (peek c))) (advance c) t)
      (erro-parse c (string-append "esperava " (symbol->string tipo)))))

(define (expect-kw c lex)
  (if (at-kw? c lex)
      (advance c)
      (erro-parse c (string-append "esperava palavra-chave '" lex "'"))))

(define (expect-pont c lex)
  (if (at-pont? c lex)
      (advance c)
      (erro-parse c (string-append "esperava '" lex "'"))))

(define (skip-newlines c)
  (if (at? c 'NEWLINE) (begin (advance c) (skip-newlines c)) #f))

### 2.3 Bloco RECEITA — metadados

```
bloco_receita ::= 'RECEITA' NEWLINE INDENT campo_meta+ DEDENT
campo_meta    ::= IDENT ':' (STRING | NUMBER | IDENT | KEYWORD) NEWLINE
```

In [ ]:
(define (parse-valor-meta c)
  (let* ((t (peek c))
         (tp (tipo-tok t)))
    (cond
      ((eq? tp 'STRING)
       (advance c) (lexema-tok t))
      ((eq? tp 'NUMBER)
       (advance c)
       (let ((n (string->number (lexema-tok t))))
         ;; pode vir unidade depois (MINUTOS, HORAS, etc.)
         (if (or (at? c 'KEYWORD) (at? c 'IDENT) (at? c 'UNIT))
             (let ((u (lexema-tok (advance c))))
               (string-append (number->string n) " " u))
             n)))
      ((or (eq? tp 'IDENT) (eq? tp 'KEYWORD))
       (advance c) (lexema-tok t))
      (else (erro-parse c "valor de metadado esperado")))))

(define (parse-receita c)
  (advance c)  ; RECEITA
  (expect c 'NEWLINE)
  (expect c 'INDENT)
  (let loop ((campos '()))
    (cond
      ((at? c 'DEDENT) (advance c) (list 'receita (reverse campos)))
      ((at? c 'NEWLINE) (advance c) (loop campos))
      (else
        (let ((nome-tok (advance c)))
          (expect-pont c ":")
          (let ((valor (parse-valor-meta c)))
            (if (at? c 'NEWLINE) (advance c) #f)
            (loop (cons (list 'campo (lexema-tok nome-tok) valor) campos))))))))

### 2.4 Bloco INGREDIENTES

```
bloco_ingredientes ::= 'INGREDIENTES' NEWLINE INDENT decl_ingrediente+ DEDENT
decl_ingrediente   ::= NUMBER UNIT (IDENT | STRING) NEWLINE
```

Se o nome vem entre aspas (STRING), trata como **sub-receita**.

In [ ]:
(define (parse-ingredientes c)
  (advance c)  ; INGREDIENTES
  (expect c 'NEWLINE)
  (expect c 'INDENT)
  (let loop ((items '()))
    (cond
      ((at? c 'DEDENT) (advance c) (list 'ingredientes (reverse items)))
      ((at? c 'NEWLINE) (advance c) (loop items))
      (else
        (let* ((qty (string->number (lexema-tok (expect c 'NUMBER))))
               (uni (lexema-tok (advance c)))
               (nom-tok (advance c))
               (nome (lexema-tok nom-tok))
               (sub? (eq? (tipo-tok nom-tok) 'STRING)))
          (if (at? c 'NEWLINE) (advance c) #f)
          (loop (cons (list 'ingrediente qty uni nome sub?) items)))))))

### 2.5 Bloco UTENSILIOS

```
bloco_utensilios ::= 'UTENSILIOS' NEWLINE INDENT decl_utensilio+ DEDENT
decl_utensilio   ::= NUMBER IDENT NEWLINE
```

In [ ]:
(define (parse-utensilios c)
  (advance c)  ; UTENSILIOS
  (expect c 'NEWLINE)
  (expect c 'INDENT)
  (let loop ((items '()))
    (cond
      ((at? c 'DEDENT) (advance c) (list 'utensilios (reverse items)))
      ((at? c 'NEWLINE) (advance c) (loop items))
      (else
        (let* ((qty (string->number (lexema-tok (expect c 'NUMBER))))
               (nom (lexema-tok (advance c))))
          (if (at? c 'NEWLINE) (advance c) #f)
          (loop (cons (list 'utensilio qty nom) items)))))))

### 2.6 Bloco METODOS — instruções

Dispatcher que escolhe entre `cmd-verbo`, `este-e`, `aguardar`,
`verificar`, `ao-mesmo-tempo` e `continuar` com base no próximo token.

**`cmd-verbo`:**
```
VERBO lista_arg ('EM' IDENT)? ('COM' IDENT)?
                ('A' (NUMBER 'GRAUS' | IDENT))?
                ('POR' NUMBER unidade_tempo)?
                ('ATE' IDENT)?
                ('NO_ESTILO' IDENT)?
                NEWLINE
lista_arg ::= arg (',' arg)*
arg       ::= IDENT ('USANDO' NUMBER UNIT)?
```

In [ ]:
;; -------- parse de uma instrução --------
(define (parse-instrucao c)
  (cond
    ((at? c 'VERB)               (parse-cmd-verbo c))
    ((at-kw? c "ESTE")           (parse-este-e c))
    ((at-kw? c "AGUARDAR")       (parse-aguardar c))
    ((at-kw? c "VERIFICAR")      (parse-verificar c))
    ((at-kw? c "AO_MESMO_TEMPO") (parse-ao-mesmo-tempo c))
    ((at-kw? c "CONTINUAR")
     (advance c)
     (if (at? c 'NEWLINE) (advance c) #f)
     (list 'continuar))
    (else (erro-parse c "instrução esperada"))))

;; bloco indentado de instruções (já consumido o NEWLINE inicial)
(define (parse-bloco-instrucoes c)
  (expect c 'INDENT)
  (let loop ((instrs '()))
    (cond
      ((at? c 'DEDENT)  (advance c) (reverse instrs))
      ((at? c 'NEWLINE) (advance c) (loop instrs))
      (else (loop (cons (parse-instrucao c) instrs))))))

;; -------- cmd-verbo --------
(define (parse-cmd-verbo c)
  (let* ((verbo (lexema-tok (advance c)))
         (args  (parse-lista-arg c))
         (em    (parse-prep-ident c "EM"))
         (com   (parse-prep-ident c "COM"))
         (a     (parse-clausula-a c))
         (por   (parse-clausula-por c))
         (ate   (parse-prep-ident c "ATE"))
         (est   (parse-prep-ident c "NO_ESTILO")))
    (if (at? c 'NEWLINE) (advance c) #f)
    (list 'verb-cmd verbo args em com a por ate est)))

(define (parse-lista-arg c)
  (if (at? c 'IDENT)
      (let loop ((args (list (parse-arg c))))
        (if (at-pont? c ",")
            (begin (advance c) (loop (cons (parse-arg c) args)))
            (reverse args)))
      '()))

(define (parse-arg c)
  (let ((nome (lexema-tok (advance c))))
    (if (at-prep? c "USANDO")
        (begin
          (advance c)
          (let ((qty (string->number (lexema-tok (advance c))))
                (uni (lexema-tok (advance c))))
            (list 'arg nome qty uni)))
        (list 'arg nome #f #f))))

(define (parse-prep-ident c lex)
  (if (at-prep? c lex)
      (begin (advance c) (lexema-tok (advance c)))
      #f))

(define (parse-clausula-a c)
  ;; 'A' NUMBER 'GRAUS' | 'A' IDENT
  (if (at-prep? c "A")
      (begin
        (advance c)
        (cond
          ((at? c 'NUMBER)
           (let ((n (string->number (lexema-tok (advance c)))))
             (if (at-kw? c "GRAUS")
                 (begin (advance c) (list n "GRAUS"))
                 (list n #f))))
          ((or (at? c 'IDENT) (at? c 'KEYWORD))
           (list #f (lexema-tok (advance c))))
          (else #f)))
      #f))

(define (parse-clausula-por c)
  ;; 'POR' NUMBER unidade_tempo
  (if (at-prep? c "POR")
      (begin
        (advance c)
        (let ((n (string->number (lexema-tok (advance c))))
              (u (lexema-tok (advance c))))
          (list n u)))
      #f))

;; -------- demais instruções --------
(define (parse-este-e c)
  (advance c)               ; ESTE
  (expect-kw c "é")
  (let ((nome (lexema-tok (expect c 'IDENT))))
    (if (at? c 'NEWLINE) (advance c) #f)
    (list 'este-e nome)))

(define (parse-aguardar c)
  (advance c)               ; AGUARDAR
  (let ((n (string->number (lexema-tok (expect c 'NUMBER))))
        (u (lexema-tok (advance c))))
    (if (at? c 'NEWLINE) (advance c) #f)
    (list 'aguardar n u)))

(define (parse-verificar c)
  (advance c)               ; VERIFICAR
  (expect-kw c "SE")
  (let ((alvo (lexema-tok (expect c 'IDENT))))
    (expect-kw c "ESTA")
    (let ((estado (lexema-tok (expect c 'IDENT))))
      (expect c 'NEWLINE)
      (let ((then-instrs (parse-bloco-instrucoes c)))
        (let ((senao-instrs
                (if (at-kw? c "SENAO")
                    (begin
                      (advance c)
                      (expect c 'NEWLINE)
                      (parse-bloco-instrucoes c))
                    #f)))
          (list 'verificar alvo estado then-instrs senao-instrs))))))

(define (parse-ao-mesmo-tempo c)
  (advance c)               ; AO_MESMO_TEMPO
  (expect c 'NEWLINE)
  (let ((instrs (parse-bloco-instrucoes c)))
    ;; v1: cada instrução do bloco é um ramo (lista de 1 instrução)
    (cons 'ao-mesmo-tempo (map (lambda (i) (list i)) instrs))))

;; -------- bloco METODOS --------
(define (parse-metodos c)
  (advance c)               ; METODOS
  (expect c 'NEWLINE)
  (let ((instrs (parse-bloco-instrucoes c)))
    (list 'metodos instrs)))

;; -------- programa --------
(define (parse-programa c)
  (skip-newlines c)
  (let* ((rec  (if (at-kw? c "RECEITA")      (parse-receita c)      #f)))
    (skip-newlines c)
    (let* ((ings (if (at-kw? c "INGREDIENTES") (parse-ingredientes c) (list 'ingredientes '()))))
      (skip-newlines c)
      (let* ((uts  (if (at-kw? c "UTENSILIOS")  (parse-utensilios c)   (list 'utensilios '()))))
        (skip-newlines c)
        (let* ((mets (if (at-kw? c "METODOS")    (parse-metodos c)      (list 'metodos '()))))
          (list 'programa rec ings uts mets))))))

(define (parse fonte)
  (parse-programa (make-cursor (tokeniza fonte))))

### 2.7 Testes do parser

Cada célula abaixo é um caso (P1–P12) de teste manual: imprime a AST
produzida para conferência visual.

In [ ]:
;; P1 — RECEITA mínima
(display "P1 — RECEITA mínima") (newline)
(display (parse "RECEITA\n    NOME: \"X\"\n"))
(newline)

In [ ]:
;; P2 — INGREDIENTES básico
(display "P2 — INGREDIENTES básico") (newline)
(display (parse "INGREDIENTES\n    2 unidade ovo\n"))
(newline)

In [ ]:
;; P3 — INGREDIENTES com sub-receita
(display "P3 — INGREDIENTES com sub-receita") (newline)
(display (parse "INGREDIENTES\n    1 unidade \"Molho\"\n"))
(newline)

In [ ]:
;; P4 — UTENSILIOS
(display "P4 — UTENSILIOS") (newline)
(display (parse "UTENSILIOS\n    1 frigideira\n"))
(newline)

In [ ]:
;; P5 — cmd-verbo simples
(display "P5 — cmd-verbo simples") (newline)
(display (parse "METODOS\n    ADICIONAR ovo EM tigela\n"))
(newline)

In [ ]:
;; P6 — lista de args + USANDO
(display "P6 — lista de args + USANDO") (newline)
(display (parse "METODOS\n    ADICIONAR ovo, sal USANDO 5 g EM tigela\n"))
(newline)

In [ ]:
;; P7 — parâmetros A GRAUS e POR MINUTOS
(display "P7 — parâmetros A GRAUS e POR MINUTOS") (newline)
(display (parse "METODOS\n    ASSAR massa EM forno A 180 GRAUS POR 40 MINUTOS\n"))
(newline)

In [ ]:
;; P8 — atribuição ESTE é
(display "P8 — atribuição ESTE é") (newline)
(display (parse "METODOS\n    MISTURAR ovo EM tigela\n    ESTE é MASSA\n"))
(newline)

In [ ]:
;; P9 — AGUARDAR
(display "P9 — AGUARDAR") (newline)
(display (parse "METODOS\n    AGUARDAR 5 MINUTOS\n"))
(newline)

In [ ]:
;; P10 — VERIFICAR SE com SENAO
(display "P10 — VERIFICAR SE com SENAO") (newline)
(display (parse "METODOS\n    VERIFICAR SE X ESTA FIRME\n        CONTINUAR\n    SENAO\n        ASSAR X EM forno\n"))
(newline)

In [ ]:
;; P11 — AO_MESMO_TEMPO com 2 ramos
(display "P11 — AO_MESMO_TEMPO com 2 ramos") (newline)
(display (parse "METODOS\n    AO_MESMO_TEMPO\n        ADICIONAR oleo EM frigideira\n        AGUARDAR 1 MINUTOS\n"))
(newline)

## 3. Pretty-printer + Validação Estática

### 3.1 Pretty-printer

Renderização recursiva por *tag*. Sem nenhum truque sofisticado: indentação
por profundidade e `display` em cada nó.

In [ ]:
(define (ident-n n)
  (if (<= n 0) "" (string-append "  " (ident-n (- n 1)))))

(define (println-prefix prefix . parts)
  (display prefix)
  (for-each (lambda (p) (display p)) parts)
  (newline))

(define (mostra-valor v)
  (cond ((string? v) (string-append "\"" v "\""))
        ((number? v) (number->string v))
        (else v)))

(define (imprime-programa p)
  (println-prefix "" "PROGRAMA")
  (let ((rec  (cadr p))
        (ings (caddr p))
        (uts  (cadddr p))
        (mets (car (cddddr p))))
    (if rec
        (imprime-receita rec 1)
        (println-prefix (ident-n 1) "(sem RECEITA)"))
    (imprime-ingredientes ings 1)
    (imprime-utensilios uts 1)
    (imprime-metodos mets 1)))

(define (imprime-receita rec n)
  (println-prefix (ident-n n) "RECEITA")
  (for-each (lambda (campo)
              (println-prefix (ident-n (+ n 1))
                              (cadr campo) " = " (mostra-valor (caddr campo))))
            (cadr rec)))

(define (imprime-ingredientes ings n)
  (println-prefix (ident-n n) "INGREDIENTES")
  (for-each (lambda (ing)
              ;; (ingrediente qty unidade nome sub?)
              (println-prefix (ident-n (+ n 1))
                              (cadr ing) " " (caddr ing) " "
                              (if (car (cddddr ing)) "<sub> " "")
                              (cadddr ing)))
            (cadr ings)))

(define (imprime-utensilios uts n)
  (println-prefix (ident-n n) "UTENSILIOS")
  (for-each (lambda (u)
              (println-prefix (ident-n (+ n 1)) (cadr u) " " (caddr u)))
            (cadr uts)))

(define (imprime-metodos mets n)
  (println-prefix (ident-n n) "METODOS")
  (for-each (lambda (i) (imprime-instrucao i (+ n 1))) (cadr mets)))

(define (imprime-instrucao i n)
  (cond
    ((eq? (car i) 'verb-cmd)       (imprime-verb-cmd i n))
    ((eq? (car i) 'este-e)         (println-prefix (ident-n n) "ESTE é " (cadr i)))
    ((eq? (car i) 'aguardar)       (println-prefix (ident-n n) "AGUARDAR " (cadr i) " " (caddr i)))
    ((eq? (car i) 'continuar)      (println-prefix (ident-n n) "CONTINUAR"))
    ((eq? (car i) 'verificar)      (imprime-verificar i n))
    ((eq? (car i) 'ao-mesmo-tempo) (imprime-ao-mesmo i n))
    (else (println-prefix (ident-n n) "(? " i ")"))))

(define (imprime-verb-cmd i n)
  ;; (verb-cmd verbo args em com a por ate estilo)
  (let ((verbo (list-ref i 1))
        (args  (list-ref i 2))
        (em    (list-ref i 3))
        (com   (list-ref i 4))
        (a     (list-ref i 5))
        (por   (list-ref i 6))
        (ate   (list-ref i 7))
        (est   (list-ref i 8)))
    (display (ident-n n))
    (display verbo) (display " ")
    (imprime-args args)
    (if em  (begin (display " EM ")  (display em))  #f)
    (if com (begin (display " COM ") (display com)) #f)
    (if a   (begin (display " A ")   (imprime-a a)) #f)
    (if por (begin (display " POR ") (display (car por)) (display " ") (display (cadr por))) #f)
    (if ate (begin (display " ATE ") (display ate)) #f)
    (if est (begin (display " NO_ESTILO ") (display est)) #f)
    (newline)))

(define (imprime-args args)
  (cond ((null? args) #f)
        (else
          (imprime-arg (car args))
          (for-each (lambda (a) (display ", ") (imprime-arg a)) (cdr args)))))

(define (imprime-arg a)
  (display (cadr a))
  (if (caddr a)
      (begin (display " USANDO ") (display (caddr a))
             (display " ") (display (cadddr a)))
      #f))

(define (imprime-a a)
  ;; a = (NUMBER "GRAUS") ou (#f IDENT)
  (cond ((and (car a) (cadr a) (string=? (cadr a) "GRAUS"))
         (display (car a)) (display " GRAUS"))
        ((cadr a) (display (cadr a)))
        ((car a) (display (car a)))
        (else #f)))

(define (imprime-verificar v n)
  ;; (verificar alvo estado then senao)
  (println-prefix (ident-n n) "VERIFICAR SE " (cadr v) " ESTA " (caddr v))
  (for-each (lambda (i) (imprime-instrucao i (+ n 1))) (cadddr v))
  (if (car (cddddr v))
      (begin
        (println-prefix (ident-n n) "SENAO")
        (for-each (lambda (i) (imprime-instrucao i (+ n 1))) (car (cddddr v))))
      #f))

(define (imprime-ao-mesmo a n)
  (println-prefix (ident-n n) "AO_MESMO_TEMPO")
  (let loop ((rs (cdr a)) (k 1))
    (cond ((null? rs) #f)
          (else
            (println-prefix (ident-n (+ n 1)) "ramo " k ":")
            (for-each (lambda (i) (imprime-instrucao i (+ n 2))) (car rs))
            (loop (cdr rs) (+ k 1))))))

### 3.2 Validação estática

Produz uma lista de `issues` no formato `(severidade código mensagem)`.

**Códigos implementados:**

| Código | Severidade | Disparador |
|---|---|---|
| `UNDECLARED_INGREDIENT` | erro | argumento de verb-cmd não declarado em INGREDIENTES nem criado por `ESTE é` |
| `UNDECLARED_UTENSIL` | erro | `EM`/`COM` referindo utensílio não declarado |
| `OVERUSE` | aviso | soma de `USANDO` de um ingrediente excede o estoque declarado |
| `UTENSIL_CONFLICT` | aviso | dois ramos de `AO_MESMO_TEMPO` referenciam o mesmo utensílio |
| `UNRESOLVED_SUBRECIPE` | aviso | sub-receita declarada (com aspas) e nunca usada |

In [ ]:
;; ---- helpers de listas (definidos aqui porque Calysto Scheme
;;      não traz `filter` como função Scheme, e algumas funções de lista
;;      vêm via interop Python) ----

(define (my-filter pred xs)
  (cond ((null? xs) '())
        ((pred (car xs))
          (cons (car xs) (my-filter pred (cdr xs))))
        (else (my-filter pred (cdr xs)))))

(define (intersect a b)
  (my-filter (lambda (x) (member x b)) a))

(define (gera-pares xs)
  (cond ((null? xs) '())
        ((null? (cdr xs)) '())
        (else
          (append
            (map (lambda (b) (list (car xs) b)) (cdr xs))
            (gera-pares (cdr xs))))))

(define (filter-sub ings)
  (my-filter (lambda (i) (car (cddddr i))) ings))

(define (filter-nao-sub ings)
  (my-filter (lambda (i) (not (car (cddddr i)))) ings))

;; -------- validação principal --------
(define (valida programa)
  (let* ((ings-list (cadr (caddr programa)))
         (uts-list  (cadr (cadddr programa)))
         (mets-list (cadr (car (cddddr programa))))
         (nomes-uts (map (lambda (u) (caddr u)) uts-list))
         (subs      (map (lambda (i) (cadddr i)) (filter-sub ings-list)))
         (estoque   (filter-nao-sub ings-list)))
    (let ((issues '())
          (nomes-disp (map (lambda (i) (cadddr i)) ings-list))
          (acc-uso '())
          (subs-usadas '()))

      (define (add-issue sev code msg)
        (set! issues (cons (list sev code msg) issues)))

      (define (decl-este nome)
        (set! nomes-disp (cons nome nomes-disp)))

      (define (mark-sub nome)
        (if (and (member nome subs) (not (member nome subs-usadas)))
            (set! subs-usadas (cons nome subs-usadas))
            #f))

      (define (chk-ing nome)
        (cond ((member nome nomes-disp) (mark-sub nome))
              (else (add-issue 'error 'UNDECLARED_INGREDIENT
                              (string-append "ingrediente '" nome "' nao declarado")))))

      (define (chk-uts nome)
        (if (or (member nome nomes-uts) (member nome nomes-disp))
            #f
            (add-issue 'error 'UNDECLARED_UTENSIL
                       (string-append "utensilio '" nome "' nao declarado"))))

      (define (registra-uso nome qty uni)
        (set! acc-uso (cons (list nome qty uni) acc-uso)))

      (define (uts-do-ramo ramo)
        (let loop ((instrs ramo) (acc '()))
          (cond
            ((null? instrs) acc)
            ((eq? (car (car instrs)) 'verb-cmd)
             (let* ((i (car instrs))
                    (em  (list-ref i 3))
                    (com (list-ref i 4)))
               (loop (cdr instrs)
                     (append (if em (list em) '())
                             (if com (list com) '())
                             acc))))
            (else (loop (cdr instrs) acc)))))

      (define (chk-conflito ramos)
        (let* ((sets (map uts-do-ramo ramos))
               (pares (gera-pares sets)))
          (for-each
            (lambda (par)
              (let ((comum (intersect (car par) (cadr par))))
                (for-each
                  (lambda (u)
                    (add-issue 'warning 'UTENSIL_CONFLICT
                               (string-append "utensilio '" u
                                 "' usado em ramos paralelos de AO_MESMO_TEMPO")))
                  comum)))
            pares)))

      (define (visit i)
        (cond
          ((eq? (car i) 'verb-cmd)
           (let ((args (list-ref i 2))
                 (em   (list-ref i 3))
                 (com  (list-ref i 4)))
             (for-each (lambda (a)
                         (chk-ing (cadr a))
                         (if (caddr a)
                             (registra-uso (cadr a) (caddr a) (cadddr a))
                             #f))
                       args)
             (if em  (chk-uts em)  #f)
             (if com (chk-uts com) #f)))
          ((eq? (car i) 'este-e)   (decl-este (cadr i)))
          ((eq? (car i) 'aguardar) #f)
          ((eq? (car i) 'continuar) #f)
          ((eq? (car i) 'verificar)
           (chk-ing (cadr i))
           (for-each visit (cadddr i))
           (if (car (cddddr i)) (for-each visit (car (cddddr i))) #f))
          ((eq? (car i) 'ao-mesmo-tempo)
           (chk-conflito (cdr i))
           (for-each (lambda (ramo) (for-each visit ramo)) (cdr i)))
          (else #f)))

      (define (chk-overuse)
        (for-each
          (lambda (ing)
            (let* ((nome    (cadddr ing))
                   (qty-est (cadr ing))
                   (uni-est (caddr ing))
                   (compat  (my-filter (lambda (u)
                                         (and (string=? (car u) nome)
                                              (string=? (caddr u) uni-est)))
                                       acc-uso))
                   (total   (apply + (map cadr compat))))
              (if (> total qty-est)
                  (add-issue 'warning 'OVERUSE
                             (string-append "ingrediente '" nome
                               "': uso total " (number->string total) " " uni-est
                               " excede estoque " (number->string qty-est) " " uni-est))
                  #f)))
          estoque))

      (define (chk-subs-nao-usadas)
        (for-each
          (lambda (s)
            (if (not (member s subs-usadas))
                (add-issue 'warning 'UNRESOLVED_SUBRECIPE
                           (string-append "sub-receita '" s
                             "' declarada mas nunca usada"))
                #f))
          subs))

      ;; execucao
      (for-each visit mets-list)
      (chk-overuse)
      (chk-subs-nao-usadas)
      (reverse issues))))

(define (imprime-relatorio issues)
  (cond
    ((null? issues)
      (display "OK — Nenhum aviso. Receita validada com sucesso.") (newline))
    (else
      (display "Relatorio (") (display (length issues)) (display " avisos):") (newline)
      (for-each
        (lambda (iss)
          (display "  [")
          (cond ((eq? (car iss) 'error)   (display "ERRO  "))
                ((eq? (car iss) 'warning) (display "AVISO "))
                (else (display "INFO  ")))
          (display "] ") (display (cadr iss))
          (display ": ") (display (caddr iss))
          (newline))
        issues))))

### 3.3 Testes da validação

Cada célula constrói uma receita-fonte que dispara um aviso específico.

In [ ]:
;; V1 — UNDECLARED_INGREDIENT (ovo não declarado)
(display "V1 — UNDECLARED_INGREDIENT (ovo não declarado)") (newline)
(imprime-relatorio (valida (parse "INGREDIENTES\n    1 unidade tigela\nMETODOS\n    ADICIONAR ovo EM tigela\n")))
(newline)

In [ ]:
;; V2 — UNDECLARED_UTENSIL (frigideira não declarada)
(display "V2 — UNDECLARED_UTENSIL (frigideira não declarada)") (newline)
(imprime-relatorio (valida (parse "INGREDIENTES\n    1 unidade ovo\nMETODOS\n    ADICIONAR ovo EM frigideira\n")))
(newline)

In [ ]:
;; V3 — OVERUSE (15 + 10 > 20)
(display "V3 — OVERUSE (15 + 10 > 20)") (newline)
(imprime-relatorio (valida (parse "INGREDIENTES\n    20 g sal\nUTENSILIOS\n    1 tigela\nMETODOS\n    ADICIONAR sal USANDO 15 g EM tigela\n    ADICIONAR sal USANDO 10 g EM tigela\n")))
(newline)

In [ ]:
;; V4 — UTENSIL_CONFLICT (dois ramos usam frigideira)
(display "V4 — UTENSIL_CONFLICT (dois ramos usam frigideira)") (newline)
(imprime-relatorio (valida (parse "INGREDIENTES\n    1 unidade ovo\n    1 colher oleo\nUTENSILIOS\n    1 frigideira\nMETODOS\n    AO_MESMO_TEMPO\n        ADICIONAR ovo EM frigideira\n        ADICIONAR oleo EM frigideira\n")))
(newline)

In [ ]:
;; V5 — UNRESOLVED_SUBRECIPE (Molho declarado e nunca usado)
(display "V5 — UNRESOLVED_SUBRECIPE (Molho declarado e nunca usado)") (newline)
(imprime-relatorio (valida (parse "INGREDIENTES\n    1 unidade ovo\n    1 unidade \"Molho\"\nUTENSILIOS\n    1 tigela\nMETODOS\n    ADICIONAR ovo EM tigela\n")))
(newline)

## 4. Driver `run` — pipeline ponta-a-ponta

Concatena tudo: `tokeniza → parse-programa → imprime-programa → valida → imprime-relatorio`.

In [ ]:
(define (run fonte)
  (let* ((tokens   (tokeniza fonte))
         (programa (parse-programa (make-cursor tokens))))
    (display "=== AST ===") (newline)
    (imprime-programa programa)
    (newline)
    (display "=== Relatório ===") (newline)
    (imprime-relatorio (valida programa))))

## 5. Exemplos selecionados

Três receitas integradas, rodando o pipeline completo `run`.

### 5.1 Omelete com Queijo (correto)

Receita canônica. Esperado: AST completa + relatório vazio.

In [ ]:
(define omelete
  "RECEITA\n    NOME: \"OmeleteComQueijo\"\n    PORCOES: 1\n    NIVEL: FACIL\n    TEMPO_PREPARO: 10 MINUTOS\n\nINGREDIENTES\n    2 unidade ovo\n    20 g sal\n    5 g pimenta_do_reino\n    1 colher oleo\n    1 unidade \"RecheioDeQueijo\"\n\nUTENSILIOS\n    1 frigideira\n    1 garfo\n    1 tigela\n    1 prato\n\nMETODOS\n    ADICIONAR ovo, sal USANDO 5 g EM tigela\n    MISTURAR ovo, pimenta_do_reino USANDO 2 g EM tigela COM garfo\n    ESTE é MISTURA01\n\n    AO_MESMO_TEMPO\n        ADICIONAR oleo EM frigideira\n        AGUARDAR 1 MINUTOS\n\n    ASSAR MISTURA01 EM frigideira A FOGO_MEDIO ATE FIRMAR\n\n    VERIFICAR SE MISTURA01 ESTA FIRME\n        CONTINUAR\n    SENAO\n        ASSAR MISTURA01 EM frigideira POR 2 MINUTOS\n\n    ADICIONAR RecheioDeQueijo EM MISTURA01\n    COLOCAR MISTURA01 EM prato")

(run omelete)

### 5.2 Omelete com erro de estoque (OVERUSE)

Mesma omelete, mas com `USANDO 50 g` de sal num estoque de 20 g.
Esperado: AST igual + 1 aviso `OVERUSE`.

In [ ]:
(define omelete-overuse
  "INGREDIENTES\n    20 g sal\n    2 unidade ovo\nUTENSILIOS\n    1 tigela\nMETODOS\n    ADICIONAR ovo, sal USANDO 50 g EM tigela")

(run omelete-overuse)

### 5.3 Conflito de utensílio (UTENSIL_CONFLICT)

Bloco `AO_MESMO_TEMPO` em que dois ramos usam `EM frigideira`.
Esperado: AST igual + 1 aviso `UTENSIL_CONFLICT`.

In [ ]:
(define omelete-conflito
  "INGREDIENTES\n    1 unidade ovo\n    1 colher oleo\nUTENSILIOS\n    1 frigideira\nMETODOS\n    AO_MESMO_TEMPO\n        ADICIONAR ovo EM frigideira\n        ADICIONAR oleo EM frigideira")

(run omelete-conflito)

## 6. Referência da gramática (EBNF)

Gramática formal da variante **RECIP-E/Indent** implementada neste notebook.
`INDENT` e `DEDENT` são tokens sintéticos emitidos pelo lexer ao crescer e
diminuir o nível de indentação (§1.4).

```
programa         ::= bloco_receita? bloco_ingredientes? bloco_utensilios? bloco_metodos?

bloco_receita    ::= 'RECEITA' NEWLINE INDENT campo_meta+ DEDENT
campo_meta       ::= IDENT ':' (STRING | NUMBER | IDENT) NEWLINE

bloco_ingredientes ::= 'INGREDIENTES' NEWLINE INDENT decl_ingrediente+ DEDENT
decl_ingrediente   ::= NUMBER unidade (IDENT | STRING) NEWLINE
unidade            ::= 'g' | 'kg' | 'ml' | 'l' | 'unidade'
                     | 'colher' | 'colher_cha' | 'xcara'
                     | 'pitada' | 'a_gosto' | 'faca'

bloco_utensilios ::= 'UTENSILIOS' NEWLINE INDENT decl_utensilio+ DEDENT
decl_utensilio   ::= NUMBER IDENT NEWLINE

bloco_metodos    ::= 'METODOS' NEWLINE INDENT instrucao+ DEDENT
instrucao        ::= cmd_verbo | atribuicao | aguardar
                   | verificar | ao_mesmo_tempo | 'CONTINUAR'

cmd_verbo        ::= VERBO lista_arg ('EM' IDENT)? ('COM' IDENT)?
                            ('A' (NUMBER 'GRAUS' | nivel_fogo))?
                            ('POR' NUMBER unidade_tempo)?
                            ('ATE' IDENT)?
                            ('NO_ESTILO' IDENT)?
                            NEWLINE
lista_arg        ::= arg (',' arg)*
arg              ::= IDENT ('USANDO' NUMBER unidade)?

atribuicao       ::= 'ESTE' 'é' IDENT NEWLINE
aguardar         ::= 'AGUARDAR' NUMBER unidade_tempo NEWLINE

verificar        ::= 'VERIFICAR' 'SE' IDENT 'ESTA' IDENT NEWLINE
                       INDENT instrucao+ DEDENT
                     ('SENAO' NEWLINE INDENT instrucao+ DEDENT)?

ao_mesmo_tempo   ::= 'AO_MESMO_TEMPO' NEWLINE
                       INDENT instrucao+ DEDENT

VERBO            ::= 'ADICIONAR' | 'MISTURAR' | 'BATER' | 'TEMPERAR'
                   | 'ASSAR' | 'FRITAR' | 'COZINHAR' | 'REFOGAR'
                   | 'GRELHAR' | 'CORTAR' | 'RALAR' | 'ESCORRER'
                   | 'COLOCAR' | 'DECORAR'

unidade_tempo    ::= 'MINUTOS' | 'HORAS' | 'SEGUNDOS'
nivel_fogo       ::= 'FOGO_ALTO' | 'FOGO_MEDIO' | 'FOGO_BAIXO'

STRING           ::= '"' <qualquer caractere exceto '"'>* '"'
NUMBER           ::= [0-9]+ ('.' [0-9]+)?
IDENT            ::= [a-zA-Z_] [a-zA-Z0-9_]*
```

### Tabela de palavras-chave

| Categoria | Palavras |
|---|---|
| Blocos | `RECEITA`, `INGREDIENTES`, `UTENSILIOS`, `METODOS` |
| Controle | `ESTE é`, `AGUARDAR`, `VERIFICAR SE`, `ESTA`, `CONTINUAR`, `SENAO`, `AO_MESMO_TEMPO` |
| Preposições | `EM`, `COM`, `USANDO`, `A`, `POR`, `ATE`, `NO_ESTILO` |
| Verbos | `ADICIONAR`, `MISTURAR`, `BATER`, `TEMPERAR`, `ASSAR`, `FRITAR`, `COZINHAR`, `REFOGAR`, `GRELHAR`, `CORTAR`, `RALAR`, `ESCORRER`, `COLOCAR`, `DECORAR` |
| Unidades | `g`, `kg`, `ml`, `l`, `unidade`, `colher`, `colher_cha`, `xcara`, `pitada`, `a_gosto`, `faca` |
| Tempo | `MINUTOS`, `HORAS`, `SEGUNDOS` |
| Temperatura | `GRAUS`, `FOGO_ALTO`, `FOGO_MEDIO`, `FOGO_BAIXO` |

> **Diferenças vs. spec PDF v1.0:** o grupo optou por delimitação por
> **indentação** em vez de `{ } ;`, e por construções `ESTE é`, `USANDO`,
> `AO_MESMO_TEMPO`, `VERIFICAR SE/SENAO`. Construções da spec não cobertas
> (`REPETIR N VEZES`, `EM_PONTO_DE`, `@CRITICO`/`@OPCIONAL`) estão listadas
> em *Trabalhos Futuros* no README.
